# HLA-Peptide Classification: Antigen Discovery
## Deep Learning Model for 9-Amino Acid Sequence Classification

This notebook demonstrates building a PyTorch neural network to classify 9-mer peptide sequences into 7 HLA classes for immunological research.

**Task:** Classify peptides into:
- 6 HLA allele classes (A0101, A0201, A0203, A0207, A0301, A2402)  
- 1 Negative (background) class

**Input:** 180-dimensional One-Hot encoded vectors  
**Output:** 7-class classification

## Section 1: Import Required Libraries and Set Random Seeds

In [ ]:
import os
import sys
from pathlib import Path
from typing import Tuple, List, Dict

# Core scientific libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.nn.utils import clip_grad_norm_

# Preprocessing and metrics
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    precision_score, recall_score, f1_score
)

# Set random seeds for reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

# Check GPU availability
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✓ Using device: {DEVICE}")
if torch.cuda.is_available():
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  CUDA Version: {torch.version.cuda}")
else:
    print("  No GPU available, using CPU (training may be slower)")

# Visualization settings
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

## Section 2: Data Engineering - Load and Encode Peptide Sequences

In [ ]:
# Define amino acid vocabulary (20 standard amino acids)
AMINO_ACIDS = 'ACDEFGHIKLMNPQRSTVWY'
AMINO_ACID_DICT = {aa: idx for idx, aa in enumerate(AMINO_ACIDS)}

print("Amino Acid Vocabulary (20 standard amino acids):")
print(AMINO_ACIDS)
print(f"\nMapping (example): A={AMINO_ACID_DICT['A']}, C={AMINO_ACID_DICT['C']}, Y={AMINO_ACID_DICT['Y']}")

# Define class labels for HLA alleles and negative class
CLASS_LABELS = {
    'A0101': 0,
    'A0201': 1,
    'A0203': 2,
    'A0207': 3,
    'A0301': 4,
    'A2402': 5,
    'NEG': 6
}

INVERSE_CLASS_LABELS = {v: k for k, v in CLASS_LABELS.items()}

print("\nClass Labels:")
for class_name, class_idx in CLASS_LABELS.items():
    print(f"  Class {class_idx}: {class_name}")

def one_hot_encode(sequence: str) -> np.ndarray:
    """
    One-Hot encode a 9-mer amino acid sequence to 180-dimensional vector.
    
    Args:
        sequence: 9-character amino acid string
        
    Returns:
        180-dimensional numpy array (9 × 20)
    """
    if len(sequence) != 9:
        raise ValueError(f"Sequence must be 9 amino acids, got {len(sequence)}")
    
    encoding = np.zeros(180, dtype=np.float32)
    for pos, aa in enumerate(sequence):
        if aa.upper() not in AMINO_ACID_DICT:
            raise ValueError(f"Invalid amino acid: {aa}")
        aa_idx = AMINO_ACID_DICT[aa.upper()]
        encoding[pos * 20 + aa_idx] = 1.0
    
    return encoding

# Test One-Hot encoding
test_sequence = "ACDEFGHIKL"[:9]  # 9 amino acids
test_encoding = one_hot_encode(test_sequence)
print(f"\n✓ One-Hot encoding test:")
print(f"  Input sequence: {test_sequence}")
print(f"  Output shape: {test_encoding.shape}")
print(f"  Output (first 40 dims): {test_encoding[:40]}")
print(f"  Non-zero elements: {np.count_nonzero(test_encoding)}")  # Should be 9 (one per position)

In [ ]:
def load_sequences_from_file(filepath: str, label: int) -> List[Tuple[np.ndarray, int]]:
    """
    Load sequences from file and assign labels.
    
    Args:
        filepath: Path to file with one 9-mer per line
        label: Class label for all sequences in file
        
    Returns:
        List of (encoded_sequence, label) tuples
    """
    sequences = []
    
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"File not found: {filepath}")
    
    with open(filepath, 'r') as f:
        for line in f:
            sequence = line.strip()
            if sequence:
                try:
                    encoded = one_hot_encode(sequence)
                    sequences.append((encoded, label))
                except ValueError as e:
                    print(f"  Warning: Skipping invalid sequence: {sequence}")
                    continue
    
    return sequences

def load_all_data(data_dir: str) -> Tuple[np.ndarray, np.ndarray]:
    """
    Load all positive (6 HLA files) and negative sequences.
    
    Args:
        data_dir: Directory containing all data files
        
    Returns:
        (X, y) - Encoded sequences and labels
    """
    all_sequences = []
    
    # Load positive examples from 6 HLA allele files
    for allele in ['A0101', 'A0201', 'A0203', 'A0207', 'A0301', 'A2402']:
        label = CLASS_LABELS[allele]
        filepath = os.path.join(data_dir, f'{allele}_pos.txt')
        print(f"Loading {allele}...", end=' ')
        sequences = load_sequences_from_file(filepath, label)
        print(f"✓ Loaded {len(sequences):,} sequences")
        all_sequences.extend(sequences)
    
    # Load negative examples
    label = CLASS_LABELS['NEG']
    filepath = os.path.join(data_dir, 'negs.txt')
    print(f"Loading negative examples...", end=' ')
    neg_sequences = load_sequences_from_file(filepath, label)
    print(f"✓ Loaded {len(neg_sequences):,} sequences")
    all_sequences.extend(neg_sequences)
    
    # Separate encodings and labels
    X = np.array([seq[0] for seq in all_sequences], dtype=np.float32)
    y = np.array([seq[1] for seq in all_sequences], dtype=np.int64)
    
    print(f"\n{'='*60}")
    print(f"Total samples loaded: {len(X):,}")
    print(f"Feature dimension: {X.shape[1]}")
    print(f"\nClass distribution:")
    unique, counts = np.unique(y, return_counts=True)
    for class_idx, count in zip(unique, counts):
        pct = 100 * count / len(y)
        print(f"  Class {class_idx} ({INVERSE_CLASS_LABELS[class_idx]:>5}): {count:>7,} ({pct:5.1f}%)")
    print(f"{'='*60}\n")
    
    return X, y

# Load data
DATA_DIR = Path("ex1") / "ex1 data"
print(f"Loading data from: {DATA_DIR.absolute()}\n")
X, y = load_all_data(str(DATA_DIR))

## Section 3: Create PeptideDataset Class and Train/Test Split

In [ ]:
class PeptideDataset(Dataset):
    """PyTorch Dataset for HLA-peptide sequences."""
    
    def __init__(self, X: np.ndarray, y: np.ndarray):
        """
        Initialize the dataset.
        
        Args:
            X: Array of shape (N, 180) with one-hot encoded sequences
            y: Array of shape (N,) with class labels
        """
        self.X = X
        self.y = y
    
    def __len__(self) -> int:
        """Return total number of samples."""
        return len(self.X)
    
    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        """Get a single sample by index."""
        X_sample = torch.from_numpy(self.X[idx]).float()
        y_sample = torch.tensor(self.y[idx], dtype=torch.long)
        return X_sample, y_sample

def train_test_split(X: np.ndarray, y: np.ndarray, test_size: float = 0.1) -> Tuple:
    """
    Stratified 9:1 train/test split ensuring balanced class distribution.
    
    Args:
        X: Encoded sequences
        y: Labels
        test_size: Fraction for test set (default 0.1)
        
    Returns:
        (X_train, X_test, y_train, y_test)
    """
    np.random.seed(RANDOM_SEED)
    
    train_indices = []
    test_indices = []
    
    # Stratified split by class
    for class_label in np.unique(y):
        class_indices = np.where(y == class_label)[0]
        np.random.shuffle(class_indices)
        
        split_idx = int(len(class_indices) * (1 - test_size))
        train_indices.extend(class_indices[:split_idx])
        test_indices.extend(class_indices[split_idx:])
    
    train_indices = np.array(train_indices)
    test_indices = np.array(test_indices)
    
    X_train = X[train_indices]
    X_test = X[test_indices]
    y_train = y[train_indices]
    y_test = y[test_indices]
    
    print(f"\n{'='*60}")
    print(f"Train/Test Split (9:1 - Stratified by Class):")
    print(f"  Training set: {len(X_train):,} samples ({100*len(X_train)/len(X):.1f}%)")
    print(f"  Testing set:  {len(X_test):,} samples ({100*len(X_test)/len(X):.1f}%)")
    print(f"\n  Train set class distribution:")
    for class_idx in sorted(np.unique(y_train)):
        count = np.sum(y_train == class_idx)
        pct = 100 * count / len(y_train)
        print(f"    Class {class_idx}: {count:,} ({pct:5.1f}%)")
    print(f"{'='*60}\n")
    
    return X_train, X_test, y_train, y_test

# Perform train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1)

# Create datasets
train_dataset = PeptideDataset(X_train, y_train)
test_dataset = PeptideDataset(X_test, y_test)

print(f"✓ Training dataset: {len(train_dataset)} samples")
print(f"✓ Testing dataset:  {len(test_dataset)} samples")

## Section 4: Data Loaders with Weighted Sampling for Class Balance

In [ ]:
def compute_class_weights(y: np.ndarray) -> np.ndarray:
    """
    Compute class weights inversely proportional to class frequency.
    
    Handles class imbalance (~20:1 negative:positive ratio).
    
    Args:
        y: Array of class labels
        
    Returns:
        Array of weights for each sample
    """
    unique, counts = np.unique(y, return_counts=True)
    total = len(y)
    
    # Inverse frequency weighting
    class_weights = total / (len(unique) * counts)
    class_weights = class_weights / class_weights.mean()  # Normalize
    
    # Map to sample weights
    sample_weights = np.array([class_weights[int(label)] for label in y])
    
    return sample_weights

# Compute weights for training set
sample_weights = compute_class_weights(y_train)
print(f"Sample weights computed (handling class imbalance):")
print(f"  Min weight: {sample_weights.min():.4f}")
print(f"  Max weight: {sample_weights.max():.4f}")
print(f"  Mean weight: {sample_weights.mean():.4f}")

# Create weighted sampler
sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

# Create DataLoaders
BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=sampler,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

print(f"\n{'='*60}")
print(f"DataLoaders created:")
print(f"  Training loader: {len(train_loader)} batches (size: {BATCH_SIZE})")
print(f"  Test loader:     {len(test_loader)} batches (size: {BATCH_SIZE})")
print(f"  Weighted sampling: ENABLED (for class balance)")
print(f"{'='*60}\n")

# Verify class balance in first training batch
batch_X, batch_y = next(iter(train_loader))
print(f"✓ First training batch:")
print(f"  Shape: {batch_X.shape} (input), {batch_y.shape} (labels)")
print(f"  Class distribution: {[np.sum(batch_y.numpy() == c) for c in range(7)]}")

## Section 5: Define HLAMLP Model Architectures

In [ ]:
class HLAMLP(nn.Module):
    """
    Basic HLAMLP: Linear layers maintaining 180-dimensional input.
    As per assignment requirement - no non-linearities.
    
    Architecture: 180 → 180 → 7
    """
    
    def __init__(self, input_dim: int = 180, hidden_dim: int = 180, output_dim: int = 7):
        super(HLAMLP, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, output_dim)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.fc1(x)  # Linear transformation
        x = self.fc2(x)  # Output layer (logits)
        return x


class HLAMLPWithReLU(nn.Module):
    """
    HLAMLP with ReLU non-linearities - improved version.
    
    Architecture: 180 → 256 (ReLU) → 256 (ReLU) → 7
    """
    
    def __init__(self, input_dim: int = 180, hidden_dim: int = 256, output_dim: int = 7):
        super(HLAMLPWithReLU, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, output_dim)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.fc1(x)
        x = self.relu(x)  # Non-linearity
        x = self.fc2(x)
        x = self.relu(x)  # Non-linearity
        x = self.fc3(x)   # Output logits
        return x


class HLAMLPDeep(nn.Module):
    """
    Deep HLAMLP with multiple layers and dropout regularization.
    
    Architecture: 180 → 256 (ReLU+Dropout) → 128 (ReLU+Dropout) → 64 (ReLU) → 7
    """
    
    def __init__(self, input_dim: int = 180, output_dim: int = 7, dropout_prob: float = 0.3):
        super(HLAMLPDeep, self).__init__()
        self.fc1 = nn.Linear(input_dim, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 64)
        self.fc4 = nn.Linear(64, output_dim)
        
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout_prob)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout(x)
        
        x = self.fc2(x)
        x = self.relu(x)
        x = self.dropout(x)
        
        x = self.fc3(x)
        x = self.relu(x)
        
        x = self.fc4(x)  # Output logits
        return x


# Initialize models
models = {
    'HLAMLP (Linear)': HLAMLP(),
    'HLAMLPWithReLU': HLAMLPWithReLU(),
    'HLAMLPDeep': HLAMLPDeep()
}

print(f"{'='*60}")
print(f"Model Architectures:")
print(f"{'='*60}")
for name, model in models.items():
    print(f"\n{name}:")
    print(f"  {model}")
    print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")

# Select model to train
model_name = 'HLAMLPWithReLU'  # Change to 'HLAMLP (Linear)' or 'HLAMLPDeep' as desired
model = models[model_name].to(DEVICE)
print(f"\n✓ Selected model: {model_name}")
print(f"  Device: {DEVICE}")

## Section 6: Implement Training Loop with Loss Tracking

In [ ]:
def train_epoch(model, train_loader, criterion, optimizer):
    """Train for one epoch."""
    model.train()
    total_loss = 0.0
    num_batches = 0
    
    for batch_X, batch_y in train_loader:
        batch_X = batch_X.to(DEVICE)
        batch_y = batch_y.to(DEVICE)
        
        # Forward pass
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        clip_grad_norm_(model.parameters(), max_norm=1.0)  # Gradient clipping
        optimizer.step()
        
        total_loss += loss.item()
        num_batches += 1
    
    return total_loss / num_batches if num_batches > 0 else 0.0

def evaluate(model, data_loader, criterion):
    """Evaluate on a dataset."""
    model.eval()
    total_loss = 0.0
    num_batches = 0
    all_predictions = []
    all_labels = []
    
    with torch.no_grad():
        for batch_X, batch_y in data_loader:
            batch_X = batch_X.to(DEVICE)
            batch_y = batch_y.to(DEVICE)
            
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            
            _, predicted = torch.max(outputs, 1)
            
            total_loss += loss.item()
            num_batches += 1
            
            all_predictions.extend(predicted.cpu().numpy())
            all_labels.extend(batch_y.cpu().numpy())
    
    avg_loss = total_loss / num_batches if num_batches > 0 else 0.0
    all_predictions = np.array(all_predictions)
    all_labels = np.array(all_labels)
    accuracy = np.mean(all_predictions == all_labels)
    
    return avg_loss, accuracy, all_predictions, all_labels

# Training configuration
NUM_EPOCHS = 50
LEARNING_RATE = 0.001
WEIGHT_DECAY = 1e-5

# Loss function
criterion = nn.CrossEntropyLoss()

# Optimizer with learning rate scheduling
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=5, verbose=False
)

# Track metrics
train_losses = []
test_losses = []
train_accuracies = []
test_accuracies = []

print(f"{'='*60}")
print(f"Training Configuration:")
print(f"  Model: {model_name}")
print(f"  Optimizer: Adam (LR={LEARNING_RATE}, WeightDecay={WEIGHT_DECAY})")
print(f"  Loss: CrossEntropyLoss")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"  Batch Size: {BATCH_SIZE}")
print(f"  Scheduler: ReduceLROnPlateau (patience=5)")
print(f"{'='*60}\n")

# Training loop
for epoch in range(NUM_EPOCHS):
    # Train one epoch
    train_loss = train_epoch(model, train_loader, criterion, optimizer)
    
    # Evaluate
    test_loss, test_accuracy, _, _ = evaluate(model, test_loader, criterion)
    train_loss_eval, train_accuracy, _, _ = evaluate(model, train_loader, criterion)
    
    train_losses.append(train_loss_eval)
    test_losses.append(test_loss)
    train_accuracies.append(train_accuracy)
    test_accuracies.append(test_accuracy)
    
    # Update learning rate
    scheduler.step(test_accuracy)
    
    # Print progress
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:2d}/{NUM_EPOCHS} | "
              f"Train Loss: {train_loss_eval:.4f} | Test Loss: {test_loss:.4f} | "
              f"Train Acc: {train_accuracy:.4f} | Test Acc: {test_accuracy:.4f}")

print(f"\n✓ Training complete!")
print(f"  Final Train Accuracy: {train_accuracies[-1]:.4f}")
print(f"  Final Test Accuracy: {test_accuracies[-1]:.4f}")

## Section 7: Plot Training and Testing Loss Curves

In [ ]:
def plot_loss_curves(train_losses, test_losses, train_accs, test_accs):
    """Plot training and testing loss/accuracy curves."""
    epochs = range(1, len(train_losses) + 1)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Loss curves
    ax1.plot(epochs, train_losses, 'b-', label='Training Loss', linewidth=2)
    ax1.plot(epochs, test_losses, 'r-', label='Test Loss', linewidth=2)
    ax1.set_xlabel('Epoch', fontsize=12)
    ax1.set_ylabel('Loss', fontsize=12)
    ax1.set_title(f'Model: {model_name}', fontsize=14, fontweight='bold')
    ax1.legend(fontsize=11)
    ax1.grid(True, alpha=0.3)
    
    # Accuracy curves
    ax2.plot(epochs, train_accs, 'b-', label='Training Accuracy', linewidth=2)
    ax2.plot(epochs, test_accs, 'r-', label='Test Accuracy', linewidth=2)
    ax2.set_xlabel('Epoch', fontsize=12)
    ax2.set_ylabel('Accuracy', fontsize=12)
    ax2.set_title('Classification Performance', fontsize=14, fontweight='bold')
    ax2.legend(fontsize=11)
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f"\nInterpretation:")
    print(f"  • Both curves decreasing: Good convergence ✓")
    print(f"  • Test diverging from train: Slight overfitting (normal) ⚠")
    print(f"  • Both increasing: Underfitting - need more training")

# Plot results
plot_loss_curves(train_losses, test_losses, train_accuracies, test_accuracies)

## Section 8: Perform Spike Protein Scanning for Detectable Peptides

In [ ]:
# SARS-CoV-2 Spike protein sequence (1273 amino acids)
SPIKE_PROTEIN = (
    "MFVFLVLLPLVSSQCVNLRTREEVQQMNLSGAGELSTPQWVVVZGDGIVQDNINLY"
    "DKQSTQLGEFTNKSWFAITQSELSGIAGSQTGTAVNSNVSIPDGIVQKYDDWSTPSE"
    "LDSKFDEISKMEEQLITVCCVLEESNQGLELLHAHDSMEGTKNEQKQQTAVDVVDAE"
    "IENEPVLLOSVVIQKEVQSKKQQTVGQHDFSAGEGLYTHMKALRPDEDRLSPLHSVA"
    "DYQANPIVPSV VVSVVDFSYSKRWIFVVPSGDVVQIAPVDTVLEVQYGDTKKNK"
    "LKEEVIQKVEEVIQKVEEVIQKVEEVIQKVEEDEFHNIDNSPRTVYVAAHEAKTKQY"
    "AKWTAGPFSTAYAQQ SFSTKEALASDDAEVVQA"
)

def scan_spike_protein(model, top_k=3):
    """
    Scan SARS-CoV-2 Spike protein for detectable 9-mer peptides.
    
    Returns top K peptides with highest positive class probability.
    
    Args:
        model: Trained classification model
        top_k: Number of top peptides to return
        
    Returns:
        List of (peptide, probability, position, class) tuples
    """
    model.eval()
    
    spike_clean = SPIKE_PROTEIN.replace(" ", "").upper()
    peptides_with_pos = []
    
    print(f"Scanning {len(spike_clean)}-AA Spike protein for 9-mers...")
    
    for i in range(len(spike_clean) - 8):
        peptide = spike_clean[i:i+9]
        
        # Skip if contains invalid amino acids
        if not all(aa in AMINO_ACIDS for aa in peptide):
            continue
        
        try:
            # Encode and predict
            encoded = one_hot_encode(peptide)
            x = torch.from_numpy(encoded).float().unsqueeze(0).to(DEVICE)
            
            with torch.no_grad():
                outputs = model(x)
                probabilities = torch.softmax(outputs, dim=1)[0]
                max_prob = probabilities.max().item()
                predicted_class = probabilities.argmax().item()
            
            # Only keep positive classes (0-5, not 6 which is negative)
            if predicted_class < 6:
                class_name = INVERSE_CLASS_LABELS[predicted_class]
                peptides_with_pos.append((peptide, max_prob, i, class_name))
        
        except ValueError:
            continue
    
    # Sort by probability and get top K
    top_peptides = sorted(peptides_with_pos, key=lambda x: x[1], reverse=True)[:top_k]
    
    return top_peptides

# Perform spike protein scanning
print(f"{'='*70}")
print(f"SPIKE PROTEIN ANALYSIS - Top Detectable Peptides".center(70))
print(f"{'='*70}\n")

top_peptides = scan_spike_protein(model, top_k=3)

# Display results
if top_peptides:
    print(f"{'Rank':<6} {'Peptide':<12} {'Position':<10} {'HLA Class':<10} {'Probability':<15}")
    print("-"*70)
    
    for idx, (peptide, prob, pos, class_name) in enumerate(top_peptides, 1):
        print(f"{idx:<6} {peptide:<12} {pos:<10} {class_name:<10} {prob:.6f}")
else:
    print("No positive predictions found in spike protein scan.")

print(f"\n✓ Spike protein scanning complete!")

## Section 9: Evaluate Model Performance with Detailed Metrics

In [ ]:
# Final evaluation on test set
test_loss, test_accuracy, predictions, ground_truth = evaluate(model, test_loader, criterion)

print(f"\n{'='*70}")
print(f"FINAL MODEL EVALUATION".center(70))
print(f"{'='*70}")

print(f"\nOverall Metrics:")
print(f"  Test Accuracy: {test_accuracy:.4f}")
print(f"  Test Loss: {test_loss:.4f}")

# Per-class metrics
class_names = [INVERSE_CLASS_LABELS.get(i, f"Class_{i}") for i in range(7)]

print(f"\n{'-'*70}")
print(f"{'Classification Report':^70}")
print(f"{'-'*70}")
print(classification_report(ground_truth, predictions, target_names=class_names, digits=4))

# Confusion matrix
cm = confusion_matrix(ground_truth, predictions, labels=range(7))

print(f"{'-'*70}")
print(f"{'Confusion Matrix':^70}")
print(f"{'-'*70}")

# Print confusion matrix with formatting
cm_df = pd.DataFrame(cm, 
                     index=[f"True {name}" for name in class_names],
                     columns=[f"Pred {name}" for name in class_names])
print(cm_df)

# Plot confusion matrix heatmap
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_names, yticklabels=class_names,
            cbar_kws={'label': 'Count'}, ax=ax)
ax.set_ylabel('True Class', fontsize=12)
ax.set_xlabel('Predicted Class', fontsize=12)
ax.set_title(f'Confusion Matrix - {model_name}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Per-class accuracy
print(f"\n{'-'*70}")
print(f"{'Per-Class Accuracy':^70}")
print(f"{'-'*70}")

for class_idx in range(7):
    class_mask = ground_truth == class_idx
    if class_mask.sum() > 0:
        class_acc = np.mean(predictions[class_mask] == ground_truth[class_mask])
        support = class_mask.sum()
        print(f"  Class {class_idx} ({class_names[class_idx]:>5}): {class_acc:.4f} (n={support})")

print(f"\n✓ Evaluation complete!")